PREREQUISITES:

Follow instructions on README.md to deploy web search stack, stock data stack, portfolio-optimization stack, and bda stack.

In [1]:
!pip install -q -r /home/sagemaker-user/amazon-bedrock-agent-samples/src/requirements.txt;

In [2]:
import sys
from pathlib import Path
import botocore

root_path = Path.cwd().parents[2]  # Go up 2 levels from your working directory
sys.path.insert(0, str(root_path))  # Insert at the beginning of sys.path

import boto3
from src.utils.bedrock_agent import (
    Agent,
    SupervisorAgent,
    Task,
    Guardrail,
    region,
    account_id,
    agents_helper
)
import argparse

from src.utils.knowledge_base_helper import KnowledgeBasesForAmazonBedrock
kb_helper = KnowledgeBasesForAmazonBedrock()

boto3 version: 1.36.3


In [3]:
sagemaker_client = boto3.client("sagemaker")
from sagemaker import get_execution_role

# Get the execution role name
execution_role_arn = get_execution_role()
print(execution_role_arn)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
arn:aws:iam::682033483837:role/service-role/AmazonSageMaker-ExecutionRole-20250213T151631


Add an inline policy (inlinePolicy.json) for the execution role above. Replace execution role and account ids

In [4]:
# Initialize boto3 client
bedrock_client = boto3.client("bedrock")
#LLM = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
LLM = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

In [5]:
def clean_up_agents():
    agents_helper.delete_agent(agent_name="portfolio_assistant", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="news_agent", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="stock_data_agent", delete_role_flag=True, verbose=True)
    agents_helper.delete_agent(agent_name="analyst_agent", delete_role_flag=True, verbose=True)
    response = bedrock_client.list_guardrails()
    for _gr in response["guardrails"]:
        if _gr["name"] == "no_bitcoin_guardrail":
            print(f"Found guardrail: {_gr['id']}")
            guardrail_identifier = _gr["id"]
            bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_identifier)

Function to delete existing agents and guardrails

Define Guardrail

In [6]:
def create_guardrail():
    return Guardrail(
        "no_bitcoin_guardrail",
        "bitcoin_topic",
        "No Bitcoin or cryptocurrency allowed in the analysis.",
        denied_topics=["bitcoin", "crypto", "cryptocurrency"],
        blocked_input_response="Sorry, this agent cannot discuss bitcoin.",
        verbose=True,
    )

Delete old agents and guardrail if they exist, create new guardrail

In [15]:
clean_up_agents()
Agent.set_force_recreate_default(True)
create_guardrail()

Found target agent, name: portfolio_assistant, id: DMIJXQZGDJ
Deleting aliases for agent DMIJXQZGDJ...
Deleting alias TBGK1HQEOI from agent DMIJXQZGDJ
Deleting alias TSTALIASID from agent DMIJXQZGDJ
Deleting agent: DMIJXQZGDJ...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_portfolio_assistant...
Found target agent, name: news_agent, id: O6LMJTYANN
Deleting aliases for agent O6LMJTYANN...
Deleting alias TSTALIASID from agent O6LMJTYANN
Deleting alias WEWYVKNLC5 from agent O6LMJTYANN
Deleting agent: O6LMJTYANN...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_news_agent...
Found target agent, name: stock_data_agent, id: 1JZYBHGC0Z
Deleting aliases for agent 1JZYBHGC0Z...
Deleting alias 52O9ADM6JP from agent 1JZYBHGC0Z
Deleting alias TSTALIASID from agent 1JZYBHGC0Z
Deleting agent: 1JZYBHGC0Z...
Deleting IAM role: AmazonBedrockExecutionRoleForAgents_stock_data_agent...
Found target agent, name: analyst_agent, id: ONNEXITVLT
Deleting aliases for agent ONNEXITVLT...
Deleti

Create stock_data subagent

In [16]:
# Define Lambda ARNs
stock_data_lookup_arn = f"arn:aws:lambda:{region}:{account_id}:function:stock_data_lookup"
portfolio_optimization_arn = f"arn:aws:lambda:{region}:{account_id}:function:FSI-PortfolioTool-BedrockAgent"

stock_data_agent = Agent.direct_create(
    name="stock_data_agent",
    role="Financial Data Collector",
    goal="Retrieve real-time and historic stock prices as well as optimizing a portfolio given tickers.",
    instructions="""Specialist in real-time financial data extraction and portfolio optimization. Use the stock_data_agent for stock price retrieval. 
                        Use the portfolio optimization action if the user requests portfolio optimization. The portfolio_optimization_action_group will always come after the stock_data_lookup if given three or more stock tickers.
                        Do not invoke the portfolio_optimization_action_group unless there are at least three tickers in the query. If asked for portfolio optimization, always
                        call the portfolio_optimization_action_group after retrieving price data from actions_stock_data_agent""",
    tools=[
        # Stock Data Lookup Tool
        {
            "code": stock_data_lookup_arn,
            "definition": {
                "name": "stock_data_lookup",
                "description": "Gets the 1-month stock price history for a given stock ticker, formatted as JSON.",
                "parameters": {
                    "ticker": {"description": "The ticker to retrieve price history for", "type": "string", "required": True}
                },
            },
        },
        # Portfolio Optimization Tool
        {
            "code": portfolio_optimization_arn,
            "definition": {
                "name": "portfolio_optimization",
                "description": "Optimizes a stock portfolio given a list of tickers and historical prices from the stock_data_lookup function.",
                "parameters": {
                    "tickers": {
                        "description": "A comma-separated list of stock tickers to include in the portfolio",
                        "type": "string",
                        "required": True
                    },
                    "prices": {
                        "description": "A JSON object with dates as keys and stock prices as values",
                        "type": "string",
                        "required": True
                    }
                }
            },
        }
    ],
    llm=LLM,
)



Deleting existing agent and corresponding lambda for: stock_data_agent...
Agent stock_data_agent not found
Creating agent stock_data_agent...
Created agent, id: WVSKABIBTL, alias id: TSTALIASID

Adding tool: stock_data_lookup...
Waiting for agent status to change. Current status CREATING
Agent id WVSKABIBTL current status: NOT_PREPARED
Adding tool: portfolio_optimization...
Waiting for agent status to change. Current status VERSIONING
Agent id WVSKABIBTL current status: PREPARED
DONE: Agent: stock_data_agent, id: WVSKABIBTL, alias id: QRKFLLCP9M



Create analyst_agent subagent

In [17]:
analyst_agent = Agent.direct_create(
            name="analyst_agent",
            role="A financial analyst specializing in synthesizing stock market trends and financial news into structured investment insights. The agent produces fact-based summaries to support strategic decision-making.",
            goal="Analyze stock trends and market news to generate insights.",
            instructions="""You are a Financial Analyst, responsible for analyzing stock trends and financial news to generate structured insights.
                            Combine stock price trends with financial news to identify key patterns.
                            Use your expertise to analyze macroeconomic indicators, company earnings, and market sentiment.
                            Ensure responses are fact-driven, clearly structured, and cite sources where applicable.
                            Do not generate financial advice—your role is to analyze and summarize available data objectively.
                            Keep analyses concise and insightful, focusing on major trends and anomalies.
                            **If given portfolio optimization pecentages, indicate that these are based on logic/math from the portfolio optimization tool, and are not considered financial advice**""",
            llm = LLM,
        )


Deleting existing agent and corresponding lambda for: analyst_agent...
Agent analyst_agent not found
Creating agent analyst_agent...
Created agent, id: QPLDD4WJNZ, alias id: TSTALIASID

Waiting for agent status to change. Current status CREATING
Agent id QPLDD4WJNZ current status: NOT_PREPARED
Waiting for agent status to change. Current status VERSIONING
Agent id QPLDD4WJNZ current status: PREPARED
DONE: Agent: analyst_agent, id: QPLDD4WJNZ, alias id: PQ2ZSXKUYI



Create news_agent subagent

In [18]:
kb_name = "financial_analysis_kb"
kb_description = "Access this knowledge base when needing to look up financial information like 10K reports, revenues, sales, net sales, loss and risks. Contains earnings calls."
kb_s3_bucket = "fsi-bda-kb"  #put your own bucket name from bda stack
kb_id, ds_id = kb_helper.create_or_retrieve_knowledge_base(
    kb_name=kb_name,
    kb_description=kb_description,
    data_bucket_name=kb_s3_bucket,
    embedding_model="amazon.titan-embed-text-v2:0",
)

# Define Web Search Lambda ARN
web_search_arn = f"arn:aws:lambda:{region}:{account_id}:function:web_search"


news_agent = Agent.direct_create(
    name="news_agent",
    role="Market News Researcher",
    goal="Fetch from the knowledge base. Then if needed, fetch latest relevant news for a given stock based on a ticker.",
    instructions=f"""1. **Check the knowledge base (ID: {kb_id}) first.**  
                   - Extract insights from **earnings calls, SEC filings (10-K, 10-Q), and corporate press releases** stored in the knowledge base.  
                   - If summarizing a 10-K or 10-Q report, **only use the web-search function if the knowledge base does not contain enough information**—always rely on the knowledge base (ID: {kb_id}).  
                    2. **If the requested information is not found in the knowledge base**, then proceed to fetch the latest financial news from the web using the `actions_news_agent` function.  
                     **Avoid unnecessary web searches** if a relevant SEC filing or report is already available in the knowledge base (ID: {kb_id}).  
                     Ensure findings are **fact-based, neutral, and structured** for investment research.""",
    tools=[
        {"code": web_search_arn, "definition": {
            "name": "web_search",
            "description": "Searches the web for investment news and earnings reports.",
            "parameters": {
                "search_query": {"description": "The query to search the web with", "type": "string", "required": True},
                "target_website": {"description": "Specific website to search", "type": "string", "required": False},
                "topic": {"description": "The topic being searched, such as 'news'", "type": "string", "required": False},
                "days": {"description": "Number of days of history to search", "type": "string", "required": False},
            },
        },
        },
         ],
    kb_id=kb_id,
    llm=LLM,
)


#try:
  #  agents_helper.associate_kb_with_agent(
   #     agent_id=news_agent.agent_id,  # Correct way to reference the agent ID
    #    kb_id=kb_id,
     #   description="Access this knowledge base when needing to analyze financial reports, including ewarnings calls, 10k, 10Q, etc."
    #)
#except botocore.exceptions.ClientError as e:
 #   if e.response["Error"]["Code"] == "ConflictException":
  #      print(f"Knowledge Base already exists. Skipping creation.")
   # else:
    #    raise  # Re-raise other unexpected errors

kb_helper.synchronize_data(kb_id, ds_id)

Knowledge Base financial_analysis_kb already exists.
Retrieved Knowledge Base Id: DSP2FBII4I
Retrieved Data Source Id: DJKTER33FD

Deleting existing agent and corresponding lambda for: news_agent...
Agent news_agent not found
Creating agent news_agent...
Created agent, id: PAKFUOWJFJ, alias id: TSTALIASID

Waiting for agent status to change. Current status CREATING
Agent id PAKFUOWJFJ current status: NOT_PREPARED
Adding tool: web_search...
Waiting for agent status to change. Current status PREPARING
Agent id PAKFUOWJFJ current status: PREPARED
Waiting for agent status to change. Current status VERSIONING
Agent id PAKFUOWJFJ current status: PREPARED
DONE: Agent: news_agent, id: PAKFUOWJFJ, alias id: VWEOR3YNLY

{ 'dataSourceId': 'DJKTER33FD',
  'ingestionJobId': '0VGU2EQIGE',
  'knowledgeBaseId': 'DSP2FBII4I',
  'startedAt': datetime.datetime(2025, 2, 28, 15, 39, 55, 967916, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 

In [19]:
portfolio_assistant = SupervisorAgent.direct_create(
    "portfolio_assistant",
    role="Portfolio Assistant",
    goal="A seasoned investment research expert responsible for orchestrating subagents to conduct a comprehensive stock analysis. This agent synthesizes market news, stock data, and analyst insights into a structured investment report.",
    collaboration_type="SUPERVISOR",
    instructions=f"""You are a Portfolio Assistant, a financial research supervisor overseeing multiple specialized agents. Your goal is to coordinate and synthesize their outputs to create a structured stock investment analysis.
                Orchestrate collaboration between subagents. Only call on a subagent if necessary for the task. Here are the subagents available:
                news_agent: Retrieves and summarizes latest financial news for a stock. Includes a knowledge base with SEC filings, earnings calls, etc.  
 **Always instruct news_agent to check the knowledge base (ID: {kb_id}) first before using the actions_news_agent function.**  
If summarizing a 10-K or 10-Q report, ensure the news_agent retrieves data **only** from the knowledge base (ID: {kb_id}) first and does not use the actions_news_agent function if the full question can be answered. If the full query cannot be satisfied, use the web search function.
                stock_data_agent: Provides historic and real-time stock prices and portfolio optimization. Portfolio optimization function that should only be used if the user asks for it specifically, and will always take price data as input.
                If prompted to optimize a portfolio, use the portflio_optimization_action_group after retrieving stock data from actions_stock_data_agent.
                analyst_agent: Synthesizes financial data and market trends into a structured, fact-based investment insight.
                Deliver a well-structured report with key observations, risks, and considerations.
                Ensure findings are comprehensive, well-organized, and relevant to investor decision-making.
                Format responses clearly, distinguishing between financial news, technical stock analysis, and synthesized insights.""",
    collaborator_agents=[
        {
            "agent": "news_agent",
            "instructions": f"Always check the knowledge base (ID: {kb_id}) first. Use this collaborator for finding news."
        },
        {
            "agent": "stock_data_agent",
            "instructions": "Use this collaborator for retrieving stock price history and performing portfolio optimization."
        },
        {
            "agent": "analyst_agent",
            "instructions": "Use this collaborator for synthesizing stock trends, financial data, and generating structured investment insights."
        }
    ],
    collaborator_objects=[news_agent, stock_data_agent, analyst_agent],
    #guardrail=no_bitcoin_guardrail,
    llm=LLM,
    verbose=False,
)


Agent portfolio_assistant not found

Created supervisor, id: HPZVKMEELQ, alias id: TSTALIASID

  associating sub-agents / collaborators to supervisor...
Waiting for agent status to change. Current status CREATING
Agent id HPZVKMEELQ current status: NOT_PREPARED
Waiting for agent status to change. Current status PREPARING
Agent id HPZVKMEELQ current status: PREPARED
Waiting for agent status to change. Current status PREPARING
Agent id HPZVKMEELQ current status: PREPARED
Waiting for agent status to change. Current status PREPARING
Agent id HPZVKMEELQ current status: PREPARED
DONE: Agent: portfolio_assistant, id: HPZVKMEELQ, alias id: F5VQC50CTD



In [ ]:
request = "what's AMZN stock price doing over the last week and relate that to recent news"
print(f"Request:\n{request}\n")
trace_level = "core"
result = portfolio_assistant.invoke(
    request,
    enable_trace=True,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

In [ ]:
request = "Optimize my portfolio with AMZN, MSFT, and GOOG, Only call on the stock data agent and analysis agent."
print(f"Request:\n{request}\n")
trace_level = "core"
result = portfolio_assistant.invoke(
    request,
    enable_trace=True,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

In [21]:
request = "Tell me about 2023 Q1 amazon earnings call."
print(f"Request:\n{request}\n")
trace_level = "core"
result = portfolio_assistant.invoke(
    request,
    enable_trace=False,
    trace_level=trace_level,
)
print(f"Final answer:\n{result}")

Request:
Tell me about 2023 Q1 amazon earnings call.

Final answer:
Amazon's Q1 2023 earnings call revealed strong financial performance with several key highlights:

Financial Performance:
• Operating income: $4.8 billion (exceeded guidance)
• Net income: $3.2 billion
• Operating cash flow: Up 38% year-over-year to $54.3 billion

AWS Segment:
• Revenue reached $21.4 billion (16% YoY growth)
• $85+ billion annualized run rate
• Some moderation in growth due to customer optimization efforts

Looking Forward:
• Q2 2023 guidance: 
  - Net sales: $127.0-133.0 billion (5-10% growth)
  - Operating income: $2.0-5.5 billion

Key Takeaways:
• Strong operational efficiency improvements
• Robust cash flow generation
• AWS remains market leader despite growth moderation
• Management maintaining cautious outlook due to macro uncertainties

The results demonstrate Amazon's resilience in challenging economic conditions while maintaining profitable growth.
